In [1]:
!pip install kokoro

In [ ]:
from kokoro import KPipeline

In [ ]:
import requests
import torch
import numpy as np
from lxml import html
from IPython.display import Audio, display


In [ ]:
pipeline = KPipeline(lang_code='a')

In [ ]:
text = "Hi, my name is holly and I'am your new assistant"
voices = ["af_bella","af_alloy","af_aoede","af_heart"]

for voice in voices:
  with torch.inference_mode():
      generator = pipeline(text, voice=voice)

      for i, (gs, ps, audio) in enumerate(generator):
          print(f"Chunk {i}: gs={gs}, ps={ps}")

          # Display audio
          display(Audio(data=audio, rate=24000, autoplay=False)) # True))


In [ ]:
r = requests.get('https://www.finalroundai.com/blog/pricing-analyst-interview-questions')

In [ ]:
# # sections
scts = [j[:-2] for i,j in zip(r.text.split('h2>')[2:-1],r.text.split('h2>')[1:-2]) if i.count('p>') >=60]
# # questions
qsts = [[j[4:-6] for j in i.split('p>')[::4]] for n,i in enumerate(r.text.split('h2>')[1:-1]) if i.count('p>') >=60]
# # problems
prob = [[j[43:-2] for j in i.split('p>')[1::4]] for n,i in enumerate(r.text.split('h2>')[1:-1]) if i.count('p>') >=60]
# # answers
ans = [[j[38:-8] for j in i.split('p>')[3::4]] for n,i in enumerate(r.text.split('h2>')[1:-1]) if i.count('p>') >=60]

In [ ]:

import html

In [ ]:
qsts = [[html.unescape(j) for j in i if len(j)>3] for i in qsts]
ans = [[html.unescape(j) for j in i if len(j)>3] for i in ans]
prob = [[html.unescape(j) for j in i if len(j)>3] for i in prob]

In [ ]:
"""
Safe Optimized Kokoro Pipeline - Better FP16 Handling
====================================================

This version includes better error handling and fallback mechanisms
for FP16/dtype issues.
"""

import torch
import numpy as np
import soundfile as sf
from kokoro import KPipeline
from IPython.display import Audio, display
from pathlib import Path
from typing import Iterator, Tuple, Optional, List
from concurrent.futures import ThreadPoolExecutor
import warnings

warnings.filterwarnings('ignore')


class SafeOptimizedKokoroPipeline:
    """Optimized Kokoro with robust FP16 handling"""

    def __init__(
        self,
        lang_code: str = 'a',
        device: Optional[str] = None,
        use_fp16: bool = False,  # Default to False for safety
        compile_model: bool = True,
        num_workers: int = 4
    ):
        """
        Initialize pipeline with safe defaults

        Args:
            lang_code: Language code
            device: Device ('cuda', 'cpu', or None for auto)
            use_fp16: Use FP16 (only if you're sure your model supports it)
            compile_model: Use torch.compile()
            num_workers: Workers for async I/O
        """
        # Auto-detect device
        if device is None:
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        else:
            self.device = device

        print(f"🚀 Initializing Kokoro pipeline on {self.device.upper()}")

        # Initialize pipeline
        self.pipeline = KPipeline(lang_code=lang_code)

        # Determine dtype
        self.use_fp16 = use_fp16 and self.device == 'cuda'
        self.dtype = torch.float16 if self.use_fp16 else torch.float32

        # Move model to device
        if hasattr(self.pipeline, 'model'):
            try:
                self.pipeline.model = self.pipeline.model.to(self.device)

                # Try FP16 if requested
                if self.use_fp16:
                    try:
                        self.pipeline.model = self.pipeline.model.to(self.dtype)
                        print("✓ Mixed precision (FP16) enabled")
                    except Exception as e:
                        print(f"⚠ FP16 failed, falling back to FP32: {e}")
                        self.use_fp16 = False
                        self.dtype = torch.float32
                        self.pipeline.model = self.pipeline.model.to(torch.float32)

                # Compile if available
                if compile_model and hasattr(torch, 'compile'):
                    try:
                        self.pipeline.model = torch.compile(
                            self.pipeline.model,
                            mode='reduce-overhead'  # Better for inference
                        )
                        print("✓ Model compiled with torch.compile()")
                    except Exception as e:
                        print(f"⚠ torch.compile() not available: {e}")

            except Exception as e:
                print(f"⚠ Model optimization failed: {e}")

        self.executor = ThreadPoolExecutor(max_workers=num_workers)
        self.sample_rate = 24000

    def generate_single(
        self,
        text: str,
        voice: str = 'af_heart',
        output_dir: Optional[Path] = None,
        autoplay_first: bool = True
    ) -> List[Tuple[int, np.ndarray]]:
        """Generate audio with automatic dtype handling"""
        results = []

        # Setup context managers
        inference_mode = torch.inference_mode()

        # Only use autocast if we're on CUDA and it makes sense
        if self.device == 'cuda':
            autocast_ctx = torch.autocast(
                device_type='cuda',
                dtype=self.dtype,
                enabled=self.use_fp16
            )
        else:
            # No-op context manager for CPU
            from contextlib import nullcontext
            autocast_ctx = nullcontext()

        with inference_mode, autocast_ctx:
            try:
                generator = self.pipeline(text, voice=voice)

                for i, (gs, ps, audio) in enumerate(generator):
                    print(f"Chunk {i}: gs={gs}, ps={ps}")

                    # Display audio
                    display(Audio(data=audio, rate=self.sample_rate, autoplay=False)) # (i==0 and autoplay_first)))

                    # Save to file asynchronously
                    if output_dir is not None:
                        output_dir = Path(output_dir)
                        output_dir.mkdir(exist_ok=True, parents=True)
                        output_path = output_dir / f'{i}.wav'
                        self.executor.submit(self._write_audio, audio, output_path)

                    results.append((i, audio))

            except RuntimeError as e:
                if "dtype" in str(e).lower():
                    print(f"\n⚠ FP16 dtype error detected. Retrying with FP32...")
                    # Disable FP16 and retry
                    self.use_fp16 = False
                    self.dtype = torch.float32
                    if hasattr(self.pipeline, 'model'):
                        self.pipeline.model = self.pipeline.model.to(torch.float32)
                    # Recursive call with FP32
                    return self.generate_single(text, voice, output_dir, autoplay_first)
                else:
                    raise

        return results

    def generate_batch(
        self,
        texts: List[str],
        voice: str = 'af_heart',
        output_dir: Optional[Path] = None
    ) -> List[List[Tuple[int, np.ndarray]]]:
        """Generate audio for multiple texts"""
        all_results = []

        print(f"🎯 Processing {len(texts)} texts in batch mode")

        for text_idx, text in enumerate(texts):
            print(f"\n📝 Text {text_idx + 1}/{len(texts)}")

            if output_dir is not None:
                text_output_dir = Path(output_dir) / f"text_{text_idx}"
            else:
                text_output_dir = None

            results = self.generate_single(
                text=text,
                voice=voice,
                output_dir=text_output_dir,
                autoplay_first=(text_idx == 0)
            )

            all_results.append(results)

        self.executor.shutdown(wait=True)
        return all_results

    def _write_audio(self, audio: np.ndarray, path: Path):
        """Write audio file"""
        try:
            sf.write(str(path), audio, self.sample_rate)
        except Exception as e:
            print(f"⚠ Error writing {path}: {e}")

    @torch.inference_mode()
    def warmup(self, text: str = "Hello world", voice: str = 'af_heart'):
        """Warmup the model"""
        print("🔥 Warming up model...")

        if self.device == 'cuda':
            autocast_ctx = torch.autocast(
                device_type='cuda',
                dtype=self.dtype,
                enabled=self.use_fp16
            )
        else:
            from contextlib import nullcontext
            autocast_ctx = nullcontext()

        with autocast_ctx:
            try:
                list(self.pipeline(text, voice=voice))
                print("✓ Warmup complete")
            except Exception as e:
                print(f"⚠ Warmup failed: {e}")


# Even simpler version - GPU only, no FP16 complications
class SimpleGPUKokoroPipeline:
    """Simplified GPU-accelerated pipeline without FP16"""

    def __init__(self, lang_code: str = 'a', num_workers: int = 4):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"🚀 Using {self.device.upper()}")

        self.pipeline = KPipeline(lang_code=lang_code)

        # Just move to GPU, no FP16
        if hasattr(self.pipeline, 'model'):
            self.pipeline.model = self.pipeline.model.to(self.device)

        self.executor = ThreadPoolExecutor(max_workers=num_workers)
        self.sample_rate = 24000

    @torch.inference_mode()
    def generate(
        self,
        text: str,
        voice: str = 'af_heart',
        output_dir: Optional[Path] = None,
        autoplay_first: bool = True
    ):
        """Generate audio - simple and reliable"""
        results = []
        generator = self.pipeline(text, voice=voice)

        for i, (gs, ps, audio) in enumerate(generator):
            print(f"Chunk {i}: gs={gs}, ps={ps}")
            display(Audio(data=audio, rate=self.sample_rate, autoplay=False)) # (i==0 and autoplay_first)))

            if output_dir is not None:
                output_dir = Path(output_dir)
                output_dir.mkdir(exist_ok=True, parents=True)
                self.executor.submit(
                    sf.write,
                    str(output_dir / f'{i}.wav'),
                    audio,
                    self.sample_rate
                )

            results.append((i, audio))

        return results


# Usage examples
def example_safe():
    """Safe optimized version"""
    pipeline = SafeOptimizedKokoroPipeline(
        lang_code='a',
        use_fp16=False,  # Start with FP32 for stability
        compile_model=True
    )

    text = "This is a test of the safe optimized pipeline."
    results = pipeline.generate_single(
        text=text,
        voice='af_heart',
        output_dir=Path('output')
    )
    print(f"\n✅ Generated {len(results)} chunks")


In [ ]:
def example_simple(texts,voice, pth):
  """Super simple GPU-only version"""
  pipeline = SimpleGPUKokoroPipeline(lang_code='a')
  for n,text in enumerate(texts):
    results = pipeline.generate(
        text=text,
        voice=voice,
        output_dir=f'simple_output/'+pth+'/'+str(n),
        autoplay_first=False
    )
    print(f"\n✅ Generated {len(results)} chunks")

In [ ]:
import os
import torchaudio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!7z a cmb1.7z cmb1.wav

In [ ]:
torchaudio.save('cmb1.wav',
torch.concat([
torch.concat([
torchaudio.load(f'simple_output/qsts_{i}/{j}/0.wav')[0],
torchaudio.load(f'simple_output/prob_{i}/{j}/0.wav')[0],
torch.concat([
    torchaudio.load(f'simple_output/ans_{i}/{j}/{k}')[0]
for k in os.listdir(f'simple_output/ans_{i}/{j}')
], axis=1),
torchaudio.load(f'simple_output/scts/{i}/0.wav')[0]
],axis=1)
for j in os.listdir(f'simple_output/ans_{i}/')
for i in range(4)
],axis=1),24000)

In [ ]:
txts = [qsts, prob, ans]
pths = ['qsts', 'prob', 'ans']
for txt,pth,voice in zip(txts,pths,voices[1:]):
  for i,j in enumerate(txt):
    example_simple(texts=j,voice=voice,pth=pth+'_'+str(i))

In [ ]:
example_simple(texts=scts,voice=voices[0],pth=f'scts')

In [ ]:
torchaudio.save('cmba.mp3',
torch.concat([
torch.concat([
torch.concat(
[torchaudio.load(f'simple_output/qsts_{i}/{j}/0.wav')[0],
torchaudio.load(f'simple_output/prob_{i}/{j}/0.wav')[0],
torch.concat(
[torchaudio.load(f'simple_output/ans_{i}/{j}/{k}')[0] for k in os.listdir(f'simple_output/ans_{i}/{j}')],
axis=1)
],
axis=1)
for j in sorted(os.listdir(f'simple_output/qsts_{i}/'))],
axis=1)
for i in range(4)],
axis=1),24000,
format="mp3",
compression=192)